# Amatrice paper experiment guide (LSTM / coherence-source comparison)

This notebook is a dedicated experiment guide for the workflow you described:

1. Crop the Amatrice interferogram pairs to `-l 42.6 42.7 -L 13.2 13.4`.
2. Generate auxiliary INT-derived products:
   - `unfilt_fine.cor`
   - `underamp_unfilt_fine.cor`
   - `underamp_unfilt_fine_circ.cor`
   - `filt_fine.std`
3. Build separate datasets for each observation file.
4. Train LSTM models with and without timestamp features.
5. Compute **non-zscore** NDI scores (Section 6) and **zscore** scores (Section 6.1).
6. Evaluate and compare all scores via AUC table (Section 8).

## Experiment design (paper hypotheses)

| Run name | Feature | Time encoding | Scoring |
|---|---|---|---|
| `filt_fine_std_difference` | phase-STD | — (pre/post NDI) | NDI |
| `filt_fine_cor_difference` | coherence | — (pre/post NDI) | NDI |
| `notime_filt_fine_std`     | phase-STD | LSTM, no time   | NDI |
| `time_filt_fine_std`       | phase-STD | LSTM + time     | NDI |
| `time_filt_fine_std_zscore`| phase-STD | LSTM + time     | z-score (probabilistic) |
| `notime_filt_fine_cor`     | coherence | LSTM, no time   | NDI |
| `time_filt_fine_cor`       | coherence | LSTM + time     | NDI |

Expected ordering (paper claims): `filt_fine_std_difference` > `filt_fine_cor_difference`;
`notime_filt_fine_std` > `filt_fine_std_difference`; `time_filt_fine_std` > `notime_filt_fine_std`;
`time_filt_fine_std_zscore` best overall.

## Temporal feature encoding (dim = 8)

The time-encoding vector for each InSAR pair is:
`[doy_sin, doy_cos, month_sin, month_cos, day_sin, day_cos, norm_interval, seq_pos]`

- **doy_sin/cos** (period 365.25): captures seasonal vegetation/snow decorrelation.
- **month_sin/cos, day_sin/cos**: fine-grained calendar position.
- **norm_interval**: temporal baseline normalised to `[0,1]`.
- **seq_pos** *(new)*: normalised sequence position `[0 = oldest, 1 = most recent/target]`.
  Provides an explicit recency cue so the LSTM can weight recent pre-event coherence
  observations more heavily.

> **Important:**
> - Coherence score uses `(pred − obs) / (pred + obs + eps)`.
> - Phase-STD score uses `(obs − pred) / (pred + obs + eps)`.
> - Therefore `filt_fine.std` must be trained/scored with `--timeseries-metric phase_std`.


In [3]:
from pathlib import Path
import json
import os
import shlex
import subprocess

BASE_DIR = Path('/scratch/yangyanchen/amatrice2025/')
GEOM_REF_DIR = Path('/scratch/yangyanchen/amatrice2025/')
CROPPED_DIR = BASE_DIR / 'cropped_paper_bbox'
EVENT_DATE = '20160824'
NEXT_DATE = '20160821_20160914'
LAT_MIN, LAT_MAX = 42.6, 42.7
LON_MIN, LON_MAX = 13.2, 13.4
PARAM_FILE = CROPPED_DIR / 'amatrice_lstm_params.json'


def run_cmd(cmd: str, env=None):
    print(f"[RUN] {cmd}")
    merged_env = os.environ.copy()
    merged_env['PYTHONUNBUFFERED'] = '1'
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    return_code = proc.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with code {return_code}')


try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'CPU'
except Exception:
    GPU_AVAILABLE = False
    DEVICE_NAME = 'CPU'

print('BASE_DIR =', BASE_DIR)
print('CROPPED_DIR =', CROPPED_DIR)
print('Training device policy = CUDA if available else CPU')
print('Detected device =', DEVICE_NAME)

print('Tip: if training fails with `epoch_loss` NameError, update your local repo copy to the fixed version before re-running.')


BASE_DIR = /scratch/yangyanchen/amatrice2025
CROPPED_DIR = /scratch/yangyanchen/amatrice2025/cropped_paper_bbox
Training device policy = CUDA if available else CPU
Detected device = NVIDIA A40
Tip: if training fails with `epoch_loss` NameError, update your local repo copy to the fixed version before re-running.


## 1. Crop the requested pairs to the paper bbox


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step crop "
    f"--base-dir {BASE_DIR} "
    f"--geom-reference-dir {GEOM_REF_DIR} "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-min {LAT_MIN} --lat-max {LAT_MAX} "
    f"--lon-min {LON_MIN} --lon-max {LON_MAX}"
)


## 2. Generate INT-derived auxiliary products


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step prepare_int_aux "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--aux-corr-win 5 --aux-phsig-win 5 "
    f"--aux-variance-win 5 --aux-variance-looks 3.0 "
    f"--aux-block-lines 512"
)


## 3. Inspect prepared products


In [4]:
run_cmd(f"find {CROPPED_DIR} -maxdepth 1 -type f | sort")


[RUN] find /scratch/yangyanchen/amatrice2025/cropped_paper_bbox -maxdepth 1 -type f | sort
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.cor
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.cor.vrt
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.cor.xml
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.hdr
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.int
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.int.vrt
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.int.xml
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_fine.std
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_underamp_unfilt_fine_circ.cor
/scratch/yangyanchen/amatrice2025/cropped_paper_bbox/20160306_20160330_filt_underamp_unfilt_fine_circ.cor.v

## 4. Build all requested datasets


In [5]:
DATASET_SPECS = [
    ('fine.cor.full', 'coherence', 'dataset_rnn_fine_cor_full'),
    ('filt_fine.cor', 'coherence', 'dataset_rnn_filt_fine_cor'),
    ('unfilt_fine.cor', 'coherence', 'dataset_rnn_unfilt_fine_cor'),
    ('underamp_unfilt_fine.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_cor'),
    ('underamp_unfilt_fine_circ.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_circ_cor'),
    ('std_cor.cor', 'coherence', 'dataset_rnn_std_cor_cor'),
    ('filt_fine.std', 'phase_std', 'dataset_rnn_filt_fine_std'),
]

for observation_file, metric, dataset_name in DATASET_SPECS:
    extra = "--looks 3.0" if observation_file == 'filt_fine.std' else ""
    run_cmd(
        f"python -m insar_pipeline.app --step build_dataset "
        f"--cropped-dir {CROPPED_DIR} "
        f"--output-dir {CROPPED_DIR} "
        f"--event-date {EVENT_DATE} "
        f"--input-source cor "
        f"--observation-file {observation_file} "
        f"--dataset-name {dataset_name} --no-legacy-aliases {extra}"
    )


[RUN] python -m insar_pipeline.app --step build_dataset --cropped-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --output-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --event-date 20160824 --input-source cor --observation-file fine.cor.full --dataset-name dataset_rnn_fine_cor_full --no-legacy-aliases 

[SARcoherenceDPM v0.3.0] [STEP] build_dataset
  - dataset_dir: /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_fine_cor_full
[RUN] python -m insar_pipeline.app --step build_dataset --cropped-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --output-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --event-date 20160824 --input-source cor --observation-file filt_fine.cor --dataset-name dataset_rnn_filt_fine_cor --no-legacy-aliases 

[SARcoherenceDPM v0.3.0] [STEP] build_dataset
/scratch/yangyanchen/amatrice2025/SARcoherenceDPM-main/insar_pipeline/dataset_builder.py:253: RuntimeWarning: divide by zero encountered in divide
  std_chunk =

## 4.5 Dataset file meanings and how to keep only necessary files\n
\n
Canonical files that are truly used by this repo are:\n
- `rnn_data.npy` / `rnn_data_std.npy` for RNN training input (metric-dependent).\n
- `score_observation.npy` / `score_observation_std.npy` for score construction.\n
- `dates.pkl` for timestamp features.\n
\n
Legacy alias files (`data.npy`, `data_std.npy`, `geninue.npy`, `geninue_std.npy`) are compatibility outputs only.\n
This notebook now uses `--no-legacy-aliases` to keep dataset folders concise.\n

## 5. LSTM training: shared hyperparameters + timestamp experiment design

- All datasets use the **same LSTM/training hyperparameters** so comparisons are fair.
- `filt_fine.std` runs `time`, `notime` (non-zscore) modes.
- `filt_fine.cor` runs **both** `time` and `notime` for a fair apples-to-apples comparison with `filt_fine.std`.
- Other coherence datasets run `notime` only.
- After this cell, **Section 5.1** adds a zscore training run for `filt_fine.std + time`,
  producing `time_filt_fine_std_zscore` for the probabilistic scoring comparison.
- Training uses GPU when CUDA is available, otherwise CPU.


In [7]:
COMMON_TRAINING_CONFIG = {
    'global': {},
    'rnn': {
        'epochs': 15,              # ← 从20增加到30（有早停保护）
        'train_batch_size': 512,
        'pred_batch_size': 1024,
        'lr': 1e-3,
        'ts_model': 'lstm',
        'optimizer': 'adamw',      # ← 改为adamw
        'weight_decay': 1e-4,      # ← 加入weight decay防过拟合
        'max_grad_norm': 1.0,
        'rnn_hidden_dim': 64,
        'rnn_num_layers': 2,
        'rnn_dropout': 0.15,       # ← 略微增加dropout
    },
}

CROPPED_DIR.mkdir(parents=True, exist_ok=True)
PARAM_FILE.write_text(json.dumps(COMMON_TRAINING_CONFIG, indent=2), encoding='utf-8')
print('Saved shared training config to', PARAM_FILE)
print(json.dumps(COMMON_TRAINING_CONFIG, indent=2))
print('Each train_predict run will echo the resolved hyperparameters and live epoch logs.')

EXPERIMENTS = [
    {
        'observation_file': 'filt_fine.std',
        'metric': 'phase_std',
        'dataset_name': 'dataset_rnn_filt_fine_std',
        'timestamp_modes': ['time', 'notime'],
    },
    {
        'observation_file': 'fine.cor.full',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_fine_cor_full',
        'timestamp_modes': ['time'],
    },
    {
        'observation_file': 'filt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_filt_fine_cor',
        'timestamp_modes': ['time'],
    },
    {
        'observation_file': 'unfilt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_unfilt_fine_cor',
        'timestamp_modes': ['time'],
    },
    {
        'observation_file': 'underamp_unfilt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_underamp_unfilt_fine_cor',
        'timestamp_modes': ['time'],
    },
    {
        'observation_file': 'underamp_unfilt_fine_circ.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_underamp_unfilt_fine_circ_cor',
        'timestamp_modes': ['time'],
    },
]

print('Experiment matrix:')
for exp in EXPERIMENTS:
    print(
        f"- {exp['observation_file']:30s} metric={exp['metric']:10s} "
        f"dataset={exp['dataset_name']} timestamp_modes={exp['timestamp_modes']}"
    )




Saved shared training config to /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/amatrice_lstm_params.json
{
  "global": {},
  "rnn": {
    "epochs": 15,
    "train_batch_size": 512,
    "pred_batch_size": 1024,
    "lr": 0.001,
    "ts_model": "lstm",
    "optimizer": "adamw",
    "weight_decay": 0.0001,
    "max_grad_norm": 1.0,
    "rnn_hidden_dim": 64,
    "rnn_num_layers": 2,
    "rnn_dropout": 0.15
  }
}
Each train_predict run will echo the resolved hyperparameters and live epoch logs.
Experiment matrix:
- filt_fine.std                  metric=phase_std  dataset=dataset_rnn_filt_fine_std timestamp_modes=['time', 'notime']
- fine.cor.full                  metric=coherence  dataset=dataset_rnn_fine_cor_full timestamp_modes=['time']
- filt_fine.cor                  metric=coherence  dataset=dataset_rnn_filt_fine_cor timestamp_modes=['time']
- unfilt_fine.cor                metric=coherence  dataset=dataset_rnn_unfilt_fine_cor timestamp_modes=['time']
- underamp_unfilt_fine.cor  

In [ ]:
for exp in EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        artifact_tag = exp['observation_file'].replace('.', '_')
        print('\n' + '=' * 80)
        print('Training run')
        print('  observation_file =', exp['observation_file'])
        print('  metric           =', exp['metric'])
        print('  dataset_dir      =', CROPPED_DIR / exp['dataset_name'])
        print('  timestamp_mode   =', timestamp_flag)
        print('  param_file       =', PARAM_FILE)
        print('  device_policy    = cuda-if-available-else-cpu')
        print('=' * 80)
        run_cmd(
            f"python -m insar_pipeline.app --step train_predict "
            f"--dataset-dir {shlex.quote(str(CROPPED_DIR / exp['dataset_name']))} "
            f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
            f"--next-date {NEXT_DATE} "
            f"--timeseries-metric {exp['metric']} "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} "
            f"--param-file {shlex.quote(str(PARAM_FILE))} {disable}"
        )

## 5.1. Zscore training: `time_filt_fine_std_zscore`

This run uses `--use-zscore` on the same `dataset_rnn_filt_fine_std` dataset.
Key differences from the plain `time_filt_fine_std` run:

- Input phase-STD values are mapped to logit-coherence space before scaling,
  giving the scaler a more Gaussian-shaped distribution to work with.
- The model is wrapped in `InSARDistributionHead`, learning both a mean **and**
  a log-variance for every pixel prediction.
- Training uses a negative log-likelihood (NLL) loss instead of MSE.

These changes allow the scoring step (Section 6.1) to divide the residual by
the predicted uncertainty, yielding a z-score anomaly map that is more
sensitive to genuine coherence loss while suppressing noisy false alarms.


In [ ]:
# Zscore training for filt_fine.std + time (produces time_filt_fine_std_zscore artifacts)
print('\n' + '=' * 80)
print('Zscore training run')
print('  observation_file = filt_fine.std')
print('  metric           = phase_std')
print('  dataset_dir      =', CROPPED_DIR / 'dataset_rnn_filt_fine_std')
print('  timestamp_mode   = time')
print('  use_zscore       = True')
print('=' * 80)
run_cmd(
    f"python -m insar_pipeline.app --step train_predict "
    f"--dataset-dir {shlex.quote(str(CROPPED_DIR / 'dataset_rnn_filt_fine_std'))} "
    f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
    f"--next-date {NEXT_DATE} "
    f"--timeseries-metric phase_std "
    f"--ts-model lstm "
    f"--artifact-tag filt_fine_std "
    f"--param-file {shlex.quote(str(PARAM_FILE))} "
    f"--use-zscore"
)



Zscore training run
  observation_file = filt_fine.std
  metric           = phase_std
  dataset_dir      = /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std
  timestamp_mode   = time
  use_zscore       = True
[RUN] python -m insar_pipeline.app --step train_predict --dataset-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std --output-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --next-date 20160821_20160914 --timeseries-metric phase_std --ts-model lstm --artifact-tag filt_fine_std --param-file /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/amatrice_lstm_params.json --use-zscore

[SARcoherenceDPM v0.3.0] [STEP] train_predict
  - dataset_dir: /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std
  - artifact_prefix: rnn_lstm_phase_std_zscore_time_filt_fine_std
  - param_file: /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/amatrice_lstm_params.json
  - param_overrides: {'epochs': 

## 6. Compute non-zscore NDI scores with the same experiment matrix

- `phase_std` score uses the phase-STD direction handled by the repository scoring logic.
- `coherence` score uses the coherence-oriented normalised difference index.
- The score loop follows the same timestamp matrix as the training loop above.
- **Section 6.1** below handles the zscore scoring for `time_filt_fine_std_zscore`.


In [ ]:
for exp in EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        artifact_tag = exp['observation_file'].replace('.', '_')
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        print('\n' + '-' * 80)
        print('Score run')
        print('  observation_file =', exp['observation_file'])
        print('  metric           =', exp['metric'])
        print('  dataset_dir      =', CROPPED_DIR / exp['dataset_name'])
        print('  timestamp_mode   =', timestamp_flag)
        print('  score_mode       = ndi')
        print('-' * 80)
        run_cmd(
            f"python -m insar_pipeline.app --step score "
            f"--dataset-dir {shlex.quote(str(CROPPED_DIR / exp['dataset_name']))} "
            f"--predict-dir {shlex.quote(str(CROPPED_DIR / 'predict'))} "
            f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
            f"--timeseries-metric {exp['metric']} "
            f"--score-mode ndi "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} {disable}"
        )


## 6.1. Zscore scoring for `time_filt_fine_std_zscore`

Uses `--score-mode zscore`: score = (genuine − predicted_mean) / predicted_std.
A large positive value means the observed phase-STD is much higher than expected
(i.e., coherence collapsed → probable damage).


In [11]:
# Zscore scoring for filt_fine.std + time + zscore
print('\n' + '-' * 80)
print('Zscore score run')
print('  observation_file = filt_fine.std')
print('  metric           = phase_std')
print('  dataset_dir      =', CROPPED_DIR / 'dataset_rnn_filt_fine_std')
print('  timestamp_mode   = time')
print('  score_mode       = zscore')
print('-' * 80)
run_cmd(
    f"python -m insar_pipeline.app --step score "
    f"--dataset-dir {shlex.quote(str(CROPPED_DIR / 'dataset_rnn_filt_fine_std'))} "
    f"--predict-dir {shlex.quote(str(CROPPED_DIR / 'predict'))} "
    f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
    f"--timeseries-metric phase_std "
    f"--score-mode zscore "
    f"--ts-model lstm "
    f"--artifact-tag filt_fine_std "
    f"--use-zscore"
)



--------------------------------------------------------------------------------
Zscore score run
  observation_file = filt_fine.std
  metric           = phase_std
  dataset_dir      = /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std
  timestamp_mode   = time
  score_mode       = zscore
--------------------------------------------------------------------------------
[RUN] python -m insar_pipeline.app --step score --dataset-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std --predict-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/predict --output-dir /scratch/yangyanchen/amatrice2025/cropped_paper_bbox --timeseries-metric phase_std --score-mode zscore --ts-model lstm --artifact-tag filt_fine_std --use-zscore

[SARcoherenceDPM v0.3.0] [STEP] score
  - dataset_dir: /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/dataset_rnn_filt_fine_std
  - predict_dir: /scratch/yangyanchen/amatrice2025/cropped_paper_bbox/predi

RuntimeError: Command failed with code 1

## 7. Optional geocoded outputs for selected scores


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step output "
    f"--predict-dir {CROPPED_DIR / 'predict'} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-file {CROPPED_DIR / 'lat_cropped.rdr'} "
    f"--lon-file {CROPPED_DIR / 'lon_cropped.rdr'} "
    f"--subset-params '-l 42.6 42.7 -L 13.2 13.4'"
)


## 8. Suggested result table export


In [ ]:
# ============================================================
# AUC comparison table
# ============================================================
# Set DAMAGE_MASK_FILE to a binary .npy or binary raster file
# where 1 = damaged pixel, 0 = undamaged pixel.
# The mask must be spatially aligned (same H×W) with the score maps.
# Example: DAMAGE_MASK_FILE = CROPPED_DIR / 'damage_mask.npy'
import numpy as np
from pathlib import Path

DAMAGE_MASK_FILE = CROPPED_DIR / 'damage_mask.npy'  # <-- set your mask path here
PREDICT_DIR = CROPPED_DIR / 'predict'

# Score-file label mapping  (file_pattern → display_name)
SCORE_LABEL_MAP = {
    'traditional_prepost_ndi_filt_fine_std.npy': 'filt_fine_std_difference',
    'traditional_prepost_ndi_filt_fine_cor.npy': 'filt_fine_cor_difference',
    'rnn_lstm_phase_std_raw_notime_filt_fine_std_score.npy': 'notime_filt_fine_std',
    'rnn_lstm_phase_std_raw_time_filt_fine_std_score.npy':   'time_filt_fine_std',
    'rnn_lstm_phase_std_zscore_time_filt_fine_std_score.npy':'time_filt_fine_std_zscore',
    'rnn_lstm_coherence_raw_notime_filt_fine_cor_score.npy':  'notime_filt_fine_cor',
    'rnn_lstm_coherence_raw_time_filt_fine_cor_score.npy':    'time_filt_fine_cor',
}

if not DAMAGE_MASK_FILE.exists():
    print(f'WARNING: damage mask not found at {DAMAGE_MASK_FILE}')
    print('Set DAMAGE_MASK_FILE above and re-run this cell to compute AUC values.')
else:
    from sklearn.metrics import roc_auc_score, f1_score, roc_curve

    damage_mask = np.load(DAMAGE_MASK_FILE).astype(np.float32)
    if damage_mask.ndim == 3:
        damage_mask = damage_mask.squeeze()
    valid = ~np.isnan(damage_mask)
    y_true = damage_mask[valid].astype(int)

    header = f"{'特征名称':<35} {'AUC值':<12} {'最佳F1-score':<16} {'阈值':<16} {'虚警率(FPR)':<12}"
    sep = '-' * 95
    print(sep)
    print(header)
    print(sep)

    for score_file, label in SCORE_LABEL_MAP.items():
        score_path = PREDICT_DIR / score_file
        if not score_path.exists():
            print(f'  {label:<33} -- score file not found: {score_file}')
            continue
        score_map = np.load(score_path).astype(np.float32)
        if score_map.ndim == 3:
            score_map = score_map.squeeze()
        # Align valid mask: skip NaN pixels in either mask or score
        combined_valid = valid & ~np.isnan(score_map)
        y_true_sub = damage_mask[combined_valid].astype(int)
        y_score_sub = score_map[combined_valid]
        if len(np.unique(y_true_sub)) < 2:
            print(f'  {label:<33} -- only one class in valid mask, skipping')
            continue
        auc = roc_auc_score(y_true_sub, y_score_sub)
        fpr_arr, tpr_arr, thresholds = roc_curve(y_true_sub, y_score_sub)
        f1_scores = []
        for thr in thresholds:
            y_pred = (y_score_sub >= thr).astype(int)
            f1_scores.append(f1_score(y_true_sub, y_pred, zero_division=0))
        best_idx = int(np.argmax(f1_scores))
        best_f1 = f1_scores[best_idx]
        best_thr = thresholds[best_idx]
        best_fpr = fpr_arr[best_idx]
        print(f'{label:<35} {auc:<12.5f} {best_f1:<16.5f} {best_thr:<16.5f} {best_fpr:<12.5f}')
    print(sep)


## 9. Traditional non-timeseries baseline (pre/post normalized difference)\n\nThis baseline does **not** use RNN/ViT. It directly computes normalized difference index from one pre-event coherence map and one post-event coherence map:\n\n`NDI = (pre - post) / (pre + post + eps)`\n\nUse this for your classical comparison experiment.\n

In [ ]:
import numpy as np
from pathlib import Path
from insar_pipeline.io_utils import read_isce_cor, write_gdal_file
from insar_pipeline.dataset_builder import calculate_std_from_cor
# calculate_std_from_cor: converts coherence γ to phase-STD σ_φ using
# σ_φ = sqrt((1 - γ²) / (2γ²)), the Cramer-Rao lower bound approximation.
# This lets us derive the phase-STD NDI baseline from the same .cor files.

PRE_PAIR = '20160809_20160821'
POST_PAIR = '20160821_20160914'
eps = 1e-6

PRE_COR_FILE  = CROPPED_DIR / f'{PRE_PAIR}_filt_fine.cor'
POST_COR_FILE = CROPPED_DIR / f'{POST_PAIR}_filt_fine.cor'
(CROPPED_DIR / 'predict').mkdir(parents=True, exist_ok=True)

pre_cor  = read_isce_cor(PRE_COR_FILE).astype(np.float32)
post_cor = read_isce_cor(POST_COR_FILE).astype(np.float32)

# -- coherence NDI baseline (filt_fine_cor_difference) --
# Higher pre-event coherence and lower post-event coherence → positive score
ndi_cor = (pre_cor - post_cor) / (pre_cor + post_cor + eps)
OUT_COR_NPY = CROPPED_DIR / 'predict' / 'traditional_prepost_ndi_filt_fine_cor.npy'
OUT_COR_BIN = CROPPED_DIR / 'predict' / 'traditional_prepost_ndi_filt_fine_cor.rdr'
np.save(OUT_COR_NPY, ndi_cor)
write_gdal_file(ndi_cor, OUT_COR_BIN)
print('Saved (coherence NDI):', OUT_COR_NPY)

# -- phase-STD NDI baseline (filt_fine_std_difference) --
# Convert coherence → phase-STD; higher post-event std and lower pre-event std → positive score
pre_std_3d  = calculate_std_from_cor(pre_cor[:, :, np.newaxis])
post_std_3d = calculate_std_from_cor(post_cor[:, :, np.newaxis])
pre_std  = pre_std_3d[:, :, 0]
post_std = post_std_3d[:, :, 0]
ndi_std = (post_std - pre_std) / (post_std + pre_std + eps)
OUT_STD_NPY = CROPPED_DIR / 'predict' / 'traditional_prepost_ndi_filt_fine_std.npy'
OUT_STD_BIN = CROPPED_DIR / 'predict' / 'traditional_prepost_ndi_filt_fine_std.rdr'
np.save(OUT_STD_NPY, ndi_std)
write_gdal_file(ndi_std, OUT_STD_BIN)
print('Saved (phase-STD NDI):', OUT_STD_NPY)

# Legacy combined file for backward compatibility
np.save(CROPPED_DIR / 'predict' / 'traditional_prepost_ndi.npy', ndi_cor)

print('Coherence NDI stats min/max/mean:',
      float(np.nanmin(ndi_cor)), float(np.nanmax(ndi_cor)), float(np.nanmean(ndi_cor)))
print('Phase-STD NDI stats min/max/mean:',
      float(np.nanmin(ndi_std)), float(np.nanmax(ndi_std)), float(np.nanmean(ndi_std)))


In [ ]:
run_cmd(f"find {CROPPED_DIR / 'predict'} -maxdepth 1 -type f | sort")
